# [7.3] Mini Activation Oracles - Exercises

Build the local mini Activation Oracle contract: activation-question batches, a tiny trained question-conditioned oracle, real baseline comparisons, OOD split reports, random-activation controls, and activation-patching checks. The CUDA report uses pinned `gelu-1l` residual activations and asks opposite questions of the same activation so activation-only probes cannot solve the task by shortcut.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter7_activation_to_language"
section = "part3_mini_activation_oracles"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_mini_activation_oracles.tests as tests

QuestionKind = Literal[
    "token",
    "code",
    "question",
    "ioi",
    "refusal",
    "truth",
    "latent_state",
]

GT_TIER = "GT-1"
EXERCISE_ID = "7.3.mini_activation_oracles"
EXPECTED_RUNTIME = "20-35 minutes for exercises; about 1 minute for CUDA preflight"
REQUIRES_GPU = False

## Activation-Question Batches

Keep activations aligned with question ids, answer ids, template ids, and the human-readable question bank.

In [ ]:
@dataclass(frozen=True)
class ActivationQuestionBatch:
    activations: t.Tensor
    question_ids: t.Tensor
    answer_ids: t.Tensor
    template_ids: t.Tensor
    questions: tuple[str, ...]


def default_activation_questions() -> tuple[str, ...]:
    raise NotImplementedError()


def build_activation_question_batch(
    activations: t.Tensor,
    question_ids: t.Tensor,
    answer_ids: t.Tensor,
    template_ids: t.Tensor,
    questions: tuple[str, ...] | None = None,
) -> ActivationQuestionBatch:
    """
    Expected shapes:
        activations: [examples, d_model]
        question_ids: [examples]
        answer_ids: [examples]
        template_ids: [examples]
    """
    raise NotImplementedError()


tests.test_build_activation_question_batch_validates_shapes_and_questions(
    build_activation_question_batch,
    default_activation_questions,
)

## Question-Conditioned Mini Oracle

Train a tiny oracle whose prediction depends on both an activation score and the question id. The same activation appears under two opposite questions, so a baseline that cannot see the question should fail.


In [ ]:
class TinyQuestionConditionedOracle(t.nn.Module):
    def __init__(self, num_questions: int = 2, hidden_dim: int = 16):
        raise NotImplementedError()

    def forward(self, scores: t.Tensor, question_ids: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


def make_question_conditioned_rows(
    residuals: t.Tensor,
    labels: t.Tensor,
    *,
    template_offset: int = 0,
) -> ActivationQuestionBatch:
    raise NotImplementedError()


def train_question_conditioned_oracle(
    batch: ActivationQuestionBatch,
    direction: t.Tensor,
    *,
    steps: int = 400,
    lr: float = 0.05,
):
    raise NotImplementedError()


def oracle_logits_for_batch(
    model: TinyQuestionConditionedOracle,
    batch: ActivationQuestionBatch,
    direction: t.Tensor,
    score_mean: t.Tensor,
    score_std: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


def train_activation_only_baseline(
    train_batch: ActivationQuestionBatch,
    eval_batch: ActivationQuestionBatch,
    direction: t.Tensor,
    score_mean: t.Tensor,
    score_std: t.Tensor,
    *,
    hidden_dim: int | None = None,
    steps: int = 300,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_question_conditioned_oracle_uses_question_ids_not_copied_probe_logits(
    make_question_conditioned_rows,
    train_question_conditioned_oracle,
    oracle_logits_for_batch,
    train_activation_only_baseline,
)


## Baseline Comparison

The oracle should beat text-only guesses and independently trained activation-only baselines. If probe logits are copied from the oracle, the question-conditioned test above should fail.


In [ ]:
@dataclass(frozen=True)
class OracleComparisonReport:
    oracle_accuracy: float
    text_only_accuracy: float
    linear_probe_accuracy: float
    mlp_probe_accuracy: float
    sae_classifier_accuracy: float
    beats_text_only: bool
    beats_or_matches_probe: bool


def _prediction_accuracy(logits: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def oracle_comparison_report(
    oracle_logits: t.Tensor,
    text_only_logits: t.Tensor,
    linear_probe_logits: t.Tensor,
    mlp_probe_logits: t.Tensor,
    sae_classifier_logits: t.Tensor,
    answer_ids: t.Tensor,
) -> OracleComparisonReport:
    raise NotImplementedError()


tests.test_oracle_comparison_report_beats_text_and_probe_baselines(
    oracle_comparison_report,
)

## OOD Splits

Report template-specific accuracy and require all OOD splits to clear the threshold.

In [ ]:
@dataclass(frozen=True)
class OODGeneralizationReport:
    heldout_template_accuracy: float
    new_name_accuracy: float
    long_context_accuracy: float
    adversarial_accuracy: float
    passes_ood: bool


def split_accuracy_by_template(
    logits: t.Tensor,
    answer_ids: t.Tensor,
    template_ids: t.Tensor,
) -> dict[int, float]:
    raise NotImplementedError()


def ood_generalization_report(
    *,
    heldout_template_logits: t.Tensor,
    heldout_template_answers: t.Tensor,
    new_name_logits: t.Tensor,
    new_name_answers: t.Tensor,
    long_context_logits: t.Tensor,
    long_context_answers: t.Tensor,
    adversarial_logits: t.Tensor,
    adversarial_answers: t.Tensor,
    min_accuracy: float = 0.75,
) -> OODGeneralizationReport:
    raise NotImplementedError()


tests.test_template_split_and_ood_reports_expose_generalization_failures(
    split_accuracy_by_template,
    ood_generalization_report,
)

## Negative Controls

A random activation should abstain or stay low-confidence. A patch should count only when it changes the answer.

In [ ]:
@dataclass(frozen=True)
class RandomActivationOracleReport:
    mean_confidence: float
    abstention_rate: float
    passes_graceful_failure: bool


@dataclass(frozen=True)
class ActivationPatchingOracleReport:
    original_answer: int
    patched_answer: int
    changed: bool


def random_activation_oracle_report(
    random_logits: t.Tensor,
    *,
    abstain_answer_id: int,
    min_abstention_rate: float = 0.5,
    max_mean_confidence: float = 0.6,
) -> RandomActivationOracleReport:
    raise NotImplementedError()


def activation_patching_oracle_report(
    original_logits: t.Tensor,
    patched_logits: t.Tensor,
) -> ActivationPatchingOracleReport:
    raise NotImplementedError()


tests.test_random_activation_report_requires_abstention_or_low_confidence(
    random_activation_oracle_report,
)
tests.test_activation_patching_report_checks_answer_change(
    activation_patching_oracle_report,
)

## Notebook Contract

After implementing the functions above, define a smoke-test wrapper. The solution notebook also checks the committed CUDA report from the pinned `gelu-1l` question-conditioned preflight.


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
